In [1]:
import pandas as pd
import numpy as np
import random
import json 

In [2]:
df = pd.read_csv('../../../data/processed/land_dataset_final_v3.csv')

In [3]:
with open('commune_nested_with_min_max.json', 'r') as file:
    data = json.load(file)

In [4]:
def get_min_max(row):
    city = row['address_subdivision']
    district = row['address_locality']
    commune = row['address_line_2']
    try:
        commune_list = data[city][district]
        for c in commune_list:
            if c['commune'] == commune:
                return pd.Series({'commune_min': c['min'], 'commune_max': c['max']})
    except KeyError:
        return pd.Series({'commune_min': None, 'commune_max': None})

df[['commune_min', 'commune_max']] = df.apply(get_min_max, axis=1)
df.dropna(subset=['commune_min', 'commune_max'], inplace=True)

# Define scoring weights
ROAD_TYPE_WEIGHTS = {
    'f_residential': 1.0, 'f_pedestrian': 0.9, 'f_cycleway': 0.85, 'f_footway': 0.8,
    'f_primary': 0.7, 'f_secondary': 0.65, 'f_tertiary': 0.6, 'f_service': 0.55,
    'f_trunk': 0.4, 'f_trunk_link': 0.4, 'f_motorway': 0.3, 'f_unclassified': 0.5,
    'f_track': 0.2, 'f_path': 0.25, 'f_steps': 0.3, 'f_disused': 0.1,
    'f_unused': 0.1, 'f_corridor': 0.15, 'f_bridleway': 0.2, 'f_road': 0.5
}

DISTANCE_DECAY = {
    'near_Vattanac_Tower_in_km': 0.7, 'near_Sisowath_Riverside_Park_in_km': 0.7,
    'near_Royal_Palace_in_km': 0.7, 'near_AEON_Mall_1_in_km': 0.6,
    'near_AEON_Mall_2_in_km': 0.6, 'near_Koh_Pich_in_km': 0.65,
    'near_Boeng_Keng_Kang_1_in_km': 0.6, 'near_Wat_Phnom_in_km': 0.55,
    'near_Chroy_Changvar_Bridge_in_km': 0.5, 'near_Phnom_Penh_Airport_in_km': 0.4,
    'near_Russian_Market_in_km': 0.45, 'near_AEON_Mall_3_in_km': 0.45,
    'near_Koh_Norea_in_km': 0.5, 'near_Camko_City_in_km': 0.45,
    'near_Olympic_Stadium_in_km': 0.4, 'near_Bassac_Lane_in_km': 0.4,
    'near_Phsar_Tmey_in_km': 0.35, 'near_Phsar_Chas_in_km': 0.35,
    'near_Phsar_kandal_in_km': 0.3
}

AMENITY_WEIGHTS = {
    'n_hospital_5km': 1.7, 'n_bank_5km': 1.6, 'n_super_market_5km': 1.5,
    'n_borey_5km': 1.5, 'n_university_5km': 1.4, 'n_secondary_school_5km': 1.3,
    'n_primary_school_5km': 1.3, 'n_pre_school_5km': 1.2, 'n_gas_station_5km': 1.2,
    'n_resturant_5km': 1.1, 'n_hotel_5km': 1.1, 'n_atm_5km': 1.0,
    'n_seven_eleven_5km': 1.0, 'n_mart_5km': 0.9, 'n_cafe_5km': 0.8
}

In [5]:
distance_columns = [col for col in df.columns if 'near_' in col and '_in_km' in col]
amenity_columns = [col for col in df.columns if col.startswith('n_') and '_5km' in col]
road_columns = [col for col in df.columns if col.startswith('f_')]

In [6]:
def calculate_desirability_score(row):
    # Distance component
    distance_scores = [np.exp(-DISTANCE_DECAY.get(col, 0.5) * row[col]) for col in distance_columns]
    distance_score = np.mean(distance_scores)
    
    # Amenity component
    amenity_scores = []
    for col in amenity_columns:
        weight = AMENITY_WEIGHTS.get(col, 1.0)
        amenity_scores.append(weight * np.log1p(row[col]))
    amenity_score = np.mean(amenity_scores) if amenity_scores else 0
    
    # Road component
    road_scores = [ROAD_TYPE_WEIGHTS[col] for col in road_columns if row[col] == 1]
    road_score = np.mean(road_scores) if road_scores else 0.5
    
    # Composite score
    composite_score = (
        0.40 * distance_score +
        0.25 * amenity_score +
        0.35 * road_score
    )
    return max(0.01, min(0.99, composite_score))

df['desirability_score'] = df.apply(calculate_desirability_score, axis=1)

In [7]:
def generate_price_per_m2(min_val, max_val, desirability_score):
    if min_val == max_val:
        base = min_val
        variation = base * random.uniform(0.15, 0.35)
        min_val = max(0, base - variation)
        max_val = base + variation
    if max_val <= min_val:
        return round(min_val, 2)
    
    total_range = max_val - min_val
    tier_factor = min(19, int(desirability_score * 20))
    premium_factor = 0.5 + 0.05 * tier_factor

    band_min = min_val + (tier_factor/20) * total_range * (0.8 + 0.02 * tier_factor)
    band_max = min_val + ((tier_factor + 1)/20) * total_range * premium_factor
    
    band_min = max(min_val, band_min)
    band_max = min(max_val, band_max)
    if band_min > band_max:
        band_min, band_max = band_max, band_min

    # Generate with micro-variations
    base_price = random.uniform(band_min, band_max)
    micro_variation = random.gauss(0, base_price * 0.08)
    price = base_price + micro_variation
    return round(max(band_min * 0.9), min(band_max * 1.2, price), 2)

In [8]:
df['price_per_m2'] = np.nan
for commune, group in df.groupby('address_line_2'):
    generated_prices = set()
    for idx, row in group.iterrows():
        price = generate_price_per_m2(
            row['commune_min'],
            row['commune_max'],
            row['desirability_score']
        )
        # Ensure cent-level uniqueness
        while price in generated_prices:
            price += random.choice([-0.01, 0.01])
        generated_prices.add(price)
        df.at[idx, 'price_per_m2'] = price

TypeError: 'float' object is not iterable

In [ ]:
df['price'] = df['price_per_m2'] * df['land_area']